In [0]:
%pip install openpyxl

In [0]:
from pathlib import Path
import pandas as pd
import openpyxl 

In [0]:

dbutils.widgets.text("raw_root", "/Volumes/workspace/raw/datos", "Raw Root")

# Léalos en Python
raw_root = Path(dbutils.widgets.get("raw_root"))

In [0]:
%sql
--DROP DATABASE IF EXISTS workspace.bronze CASCADE; 
drop table if exists workspace.bronze.tbl_representantes;
drop table if exists workspace.bronze.tbl_productos;
drop table if exists workspace.bronze.tbl_ventas_detalle;




In [0]:
%sql
CREATE DATABASE IF NOT EXISTS workspace.bronze
COMMENT 'Capa Bronze: datos crudos procesados'

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.bronze.tbl_representantes (
  `Representante` STRING,
  `Ciudad` STRING, 
  `Fotografía` DOUBLE
)

""")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.bronze.tbl_productos (
    `CódigoProducto` STRING,
    `Descripción` STRING,
    `Precio de venta` DOUBLE,
    `Costo de venta` BIGINT,
    `Almacen` BIGINT,
    `Vendidos` BIGINT
)USING DELTA
TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name'
)

""")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.bronze.tbl_ventas_detalle (
  `Fecha` TIMESTAMP,
  `Representante` STRING,
  `CódigoProducto` STRING,
  `Unidades` BIGINT
)

""")

In [0]:

archivo_excel =raw_root / Path("DatosDeVentaDeTiendaDeTelefonos(10milDatos).xlsx")

excel = pd.ExcelFile(archivo_excel)

print(excel.sheet_names)

In [0]:
df =  pd.read_excel(
    archivo_excel,
    sheet_name="Hoja2"
)
df_spark = spark.createDataFrame(df)

In [0]:
# Usamos overwrite para reemplazar completamente los datos existentes
df_spark.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.bronze.tbl_representantes")

In [0]:
df =  pd.read_excel(
    archivo_excel,
    sheet_name="Hoja3"
)

df_spark = spark.createDataFrame(df)

In [0]:
# Usamos overwrite para reemplazar completamente los datos existentes
df_spark.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.bronze.tbl_productos")

In [0]:
df =  pd.read_excel(
    archivo_excel,
    sheet_name="Hoja1"
)

df_spark = spark.createDataFrame(df)

In [0]:
# Usamos overwrite para reemplazar completamente los datos existentes
df_spark.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.bronze.tbl_ventas_detalle")

In [0]:
%sql
use catalog `workspace`; select * from `bronze`.`tbl_ventas_detalle` limit 100;